In [0]:
# 0. Widget: user sets/amends this before running the notebook
dbutils.widgets.text("week", "4", "Week")
week_value = dbutils.widgets.get("week")

In [0]:
catalog = "dev"
db_name = "epl_26"


base_dir_data = spark.sql("DESCRIBE EXTERNAL LOCATION `data-zone`").select("url").collect()[0][0]
#display(base_dir_data)

landing_zone = base_dir_data + "/raw/epl_26_27"
landing_zone

'abfss://sbit-unmanaged-dev@opsygotti.dfs.core.windows.net/data-zone/raw/epl_26_27'

In [0]:
import time

# --- Configuration ---

start = int(time.time())

# ==- CREATE DATABASE ===
print(f"Creating database {catalog}.{db_name})...", end='')
spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog}.{db_name}")
print("Done")

Creating database dev.epl_26)...Done


In [0]:
%sql
DROP TABLE IF EXISTS dev.epl_26.week_4_table

In [0]:
print("Creating epl_26 table..", end='')
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.week_{week_value}_table_bz (
        week string,
        position string,
        team string,
        logo string,
        played string,
        won string,
        drawn string,
        lost string,
        goals_for string,
        goals_against string,
        goal_difference string,
        points string
    )
""")
print("Done")

Creating epl_26 table..Done


In [0]:
from pyspark.sql import functions as F

week = 4

# 2. Read it as CSV — the .json extension is misleading, content is delimited text
raw_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("quote", '"')
    .option("escape", '"')
    .load(f"abfss://sbit-unmanaged-dev@opsygotti.dfs.core.windows.net/data-zone/raw/epl_26_27/week_{week_value}_1.json")
)

raw_df.printSchema()   # sanity check: you should see 0_position, 0_team, ... 19_points, timezone, status

root
 |-- 0_position: string (nullable = true)
 |-- 0_team: string (nullable = true)
 |-- 0_logo: string (nullable = true)
 |-- 0_played: string (nullable = true)
 |-- 0_won: string (nullable = true)
 |-- 0_drawn: string (nullable = true)
 |-- 0_lost: string (nullable = true)
 |-- 0_goals_for: string (nullable = true)
 |-- 0_goals_against: string (nullable = true)
 |-- 0_goal_difference: string (nullable = true)
 |-- 0_points: string (nullable = true)
 |-- 1_position: string (nullable = true)
 |-- 1_team: string (nullable = true)
 |-- 1_logo: string (nullable = true)
 |-- 1_played: string (nullable = true)
 |-- 1_won: string (nullable = true)
 |-- 1_drawn: string (nullable = true)
 |-- 1_lost: string (nullable = true)
 |-- 1_goals_for: string (nullable = true)
 |-- 1_goals_against: string (nullable = true)
 |-- 1_goal_difference: string (nullable = true)
 |-- 1_points: string (nullable = true)
 |-- 2_position: string (nullable = true)
 |-- 2_team: string (nullable = true)
 |-- 2_logo: 

In [0]:
# 3. Reshape wide -> long (20 teams per row -> 20 rows)
team_structs = [
    F.struct(
        F.col(f"{i}_position").alias("position"),
        F.col(f"{i}_team").alias("team"),
        F.col(f"{i}_logo").alias("logo"),
        F.col(f"{i}_played").alias("played"),
        F.col(f"{i}_won").alias("won"),
        F.col(f"{i}_drawn").alias("drawn"),
        F.col(f"{i}_lost").alias("lost"),
        F.col(f"{i}_goals_for").alias("goals_for"),
        F.col(f"{i}_goals_against").alias("goals_against"),
        F.col(f"{i}_goal_difference").alias("goal_difference"),
        F.col(f"{i}_points").alias("points"),
    )
    for i in range(20)
]

# The below was valid before I include the week value. 
long_df = (
    raw_df
    .select(F.explode(F.array(*team_structs)).alias("t"))
    .select("t.*")
    .withColumn("week", F.lit(week_value))
    .select("week", "position", "team", "logo", "played", "won", "drawn",
            "lost", "goals_for", "goals_against", "goal_difference", "points")
)

display(long_df)



week,position,team,logo,played,won,drawn,lost,goals_for,goals_against,goal_difference,points
5,1,Manchester City,https://e0.365dm.com/football/badges/64/345.png,4,4,0,0,8,2,+6,12
5,2,Arsenal,https://e0.365dm.com/football/badges/64/413.png,5,4,0,1,8,4,+4,12
5,3,Brighton and Hove Albion,https://e0.365dm.com/football/badges/64/212.png,5,3,1,1,16,5,+11,10
5,4,Brentford,https://e0.365dm.com/football/badges/64/194.png,5,2,3,0,10,4,+6,9
5,5,Everton,https://e0.365dm.com/football/badges/64/229.png,5,2,3,0,6,3,+3,9
5,6,Leeds United,https://e0.365dm.com/football/badges/64/183.png,4,2,2,0,7,3,+4,8
5,7,Hull City,https://e0.365dm.com/football/badges/64/253.png,5,2,2,1,6,4,+2,8
5,8,Newcastle United,https://e0.365dm.com/football/badges/64/409.png,5,2,2,1,9,9,0,8
5,9,Chelsea,https://e0.365dm.com/football/badges/64/524.png,5,2,1,2,10,12,-2,7
5,10,Liverpool,https://e0.365dm.com/football/badges/64/155.png,4,1,3,0,6,4,+2,6


In [0]:
long_df.write.mode("append").insertInto(f"{catalog}.{db_name}.week_{week_value}_table_bz")

In [0]:
display(spark.sql(f"SELECT * FROM {catalog}.{db_name}.week_{week_value}_table_bz"))

week,position,team,logo,played,won,drawn,lost,goals_for,goals_against,goal_difference,points
5,1,Manchester City,https://e0.365dm.com/football/badges/64/345.png,4,4,0,0,8,2,+6,12
5,2,Arsenal,https://e0.365dm.com/football/badges/64/413.png,5,4,0,1,8,4,+4,12
5,3,Brighton and Hove Albion,https://e0.365dm.com/football/badges/64/212.png,5,3,1,1,16,5,+11,10
5,4,Brentford,https://e0.365dm.com/football/badges/64/194.png,5,2,3,0,10,4,+6,9
5,5,Everton,https://e0.365dm.com/football/badges/64/229.png,5,2,3,0,6,3,+3,9
5,6,Leeds United,https://e0.365dm.com/football/badges/64/183.png,4,2,2,0,7,3,+4,8
5,7,Hull City,https://e0.365dm.com/football/badges/64/253.png,5,2,2,1,6,4,+2,8
5,8,Newcastle United,https://e0.365dm.com/football/badges/64/409.png,5,2,2,1,9,9,0,8
5,9,Chelsea,https://e0.365dm.com/football/badges/64/524.png,5,2,1,2,10,12,-2,7
5,10,Liverpool,https://e0.365dm.com/football/badges/64/155.png,4,1,3,0,6,4,+2,6


In [0]:
print("Creating epl_26 table..", end='')
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.week_{week_value}_results_bz (
        week string,
        match_date string,
        team_name string,
        opponent string,
        venue string,
        result_score string,
        result string,
        goal_scorers string,
        goal_times string,
        form_last_5 string
    )
""")
print("Done")

Creating epl_26 table..Done


In [0]:
from pyspark.sql import functions as F


# 2. Read it as CSV — the .json extension is misleading, content is delimited text
result_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("quote", '"')
    .option("escape", '"')
    .load(f"abfss://sbit-unmanaged-dev@opsygotti.dfs.core.windows.net/data-zone/raw/epl_26_27/premier_league_matchday{week_value}.csv")
)

result_df.printSchema()   # sanity check: you should see 0_position, 0_team, ... 19_points, timezone, status

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7654708469926362>, line 11
      1 from pyspark.sql import functions as F
      4 # 2. Read it as CSV — the .json extension is misleading, content is delimited text
      5 result_df = (
      6     spark.read
      7     .format("csv")
      8     .option("header", "true")
      9     .option("quote", '"')
     10     .option("escape", '"')
---> 11     .load(f"abfss://sbit-unmanaged-dev@opsygotti.dfs.core.windows.net/data-zone/raw/epl_26_27/premier_league_matchday{week_value}.csv")
     12 )
     14 result_df.printSchema()   # sanity check: you should see 0_position, 0_team, ... 19_points, timezone, status

File /databricks/spark/python/pyspark/databricks/instrumentation/instrumentation_utils.py:217, in _wrap_function.<locals>.wrapper(*args, **kwargs)
    215 start = time.perf_counter()
    216 try:
--> 217     res = func

In [0]:
#df_results.printSchema()
results_df = (
    result_df.select("matchweek", "match_date", "team_name", "opponent", "venue", "result_score", "result", "goal_scorers", "goal_times", "form_last_5")
)


display(result_df)   # should show 20 rows, one per team

matchweek,match_date,team_name,opponent,venue,result_score,result,goal_scorers,goal_times,form_last_5
3,2026-09-06,Arsenal,Chelsea,H,2-1,W,Havertz 20'; Odegaard 50',20'; 50',WWW
3,2026-09-05,Aston Villa,Hull City,A,0-0,D,-,-,LLD
3,2026-09-05,Bournemouth,Newcastle United,A,2-2,D,Tavernier 9'; Thiaw (OG) 35',9'; 35',LDD
3,2026-09-05,Brentford,Sunderland,H,1-1,D,Janelt 57',57',WDD
3,2026-09-05,Brighton,Leeds United,H,1-1,D,Vuskovic 71',71',WLD
3,2026-09-06,Chelsea,Arsenal,A,1-2,L,Rogers 1',1',WWL
3,2026-09-05,Coventry City,Manchester City,A,0-1,L,-,-,LLL
3,2026-09-05,Crystal Palace,Fulham,A,3-2,W,Mitchell 35'; Mitchell 54'; Chilwell 77',35'; 54'; 77',LLW
3,2026-09-06,Everton,Manchester United,H,2-2,D,George 84'; Maitland-Niles 90',84'; 90',WDD
3,2026-09-05,Fulham,Crystal Palace,H,2-3,L,King 11'; Palacios 42',11'; 42',LLL


In [0]:
results_df.write.mode("append").insertInto(f"{catalog}.{db_name}.week_{week_value}_results_bz")

In [0]:
display(spark.read.table(f"{catalog}.{db_name}.week_{week_value}_results_bz"))

week,match_date,team_name,opponent,venue,result_score,result,goal_scorers,goal_times,form_last_5
3,2026-09-06,Arsenal,Chelsea,H,2-1,W,Havertz 20'; Odegaard 50',20'; 50',WWW
3,2026-09-05,Aston Villa,Hull City,A,0-0,D,-,-,LLD
3,2026-09-05,Bournemouth,Newcastle United,A,2-2,D,Tavernier 9'; Thiaw (OG) 35',9'; 35',LDD
3,2026-09-05,Brentford,Sunderland,H,1-1,D,Janelt 57',57',WDD
3,2026-09-05,Brighton,Leeds United,H,1-1,D,Vuskovic 71',71',WLD
3,2026-09-06,Chelsea,Arsenal,A,1-2,L,Rogers 1',1',WWL
3,2026-09-05,Coventry City,Manchester City,A,0-1,L,-,-,LLL
3,2026-09-05,Crystal Palace,Fulham,A,3-2,W,Mitchell 35'; Mitchell 54'; Chilwell 77',35'; 54'; 77',LLW
3,2026-09-06,Everton,Manchester United,H,2-2,D,George 84'; Maitland-Niles 90',84'; 90',WDD
3,2026-09-05,Fulham,Crystal Palace,H,2-3,L,King 11'; Palacios 42',11'; 42',LLL


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import from_csv, col
from pyspark.sql.window import Window

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.week_{week_value}_table (
    week string,
    position int,
    team string,
    logo string,
    played int,
    won int,
    drawn int,
    lost int,
    goals_for int,
    goals_against int,
    goal_difference int,
    points int
) USING DELTA
""")

DataFrame[]

In [0]:
table_clean = (spark.read.table(f"{catalog}.{db_name}.week_{week_value}_table_bz")
        .withColumn("position", F.col("position").cast("int"))
        .withColumn("won", F.col("won").cast("int"))
        .withColumn("drawn", F.col("drawn").cast("int"))
        .withColumn("lost", F.col("lost").cast("int"))
        .withColumn("goals_for", F.col("goals_for").cast("int"))
        .withColumn("goals_against", F.col("goals_against").cast("int"))
        .withColumn("goal_difference", F.col("goal_difference").cast("int"))
        .withColumn("points", F.col("points").cast("int"))
        .select(
             "week",
             "position",
             "team",
             "logo",
             "played",
             "won",
             "drawn",
             "lost",
             "goals_for",
             "goals_against",
             "goal_difference",
             "points"
        )
        .dropDuplicates(["team"])
)

display(table_clean)


week,position,team,logo,played,won,drawn,lost,goals_for,goals_against,goal_difference,points
5,2,Arsenal,https://e0.365dm.com/football/badges/64/413.png,5,4,0,1,8,4,4,12
5,15,Aston Villa,https://e0.365dm.com/football/badges/64/238.png,5,1,1,3,4,9,-5,4
5,16,Bournemouth,https://e0.365dm.com/football/badges/64/333.png,4,0,3,1,6,7,-1,3
5,4,Brentford,https://e0.365dm.com/football/badges/64/194.png,5,2,3,0,10,4,6,9
5,3,Brighton and Hove Albion,https://e0.365dm.com/football/badges/64/212.png,5,3,1,1,16,5,11,10
5,9,Chelsea,https://e0.365dm.com/football/badges/64/524.png,5,2,1,2,10,12,-2,7
5,18,Coventry City,https://e0.365dm.com/football/badges/64/571.png,5,1,0,4,1,10,-9,3
5,17,Crystal Palace,https://e0.365dm.com/football/badges/64/234.png,4,1,0,3,6,11,-5,3
5,5,Everton,https://e0.365dm.com/football/badges/64/229.png,5,2,3,0,6,3,3,9
5,20,Fulham,https://e0.365dm.com/football/badges/64/407.png,4,0,1,3,4,7,-3,1


In [0]:
table_clean.createOrReplaceTempView("table")

spark.sql(f"""
MERGE INTO {catalog}.{db_name}.week_{week_value}_table t
USING table s
ON t.team = s.team
WHEN NOT MATCHED THEN INSERT *
""")

display(spark.table(f"{catalog}.{db_name}.week_{week_value}_table").limit(10))

week,position,team,logo,played,won,drawn,lost,goals_for,goals_against,goal_difference,points
5,2,Arsenal,https://e0.365dm.com/football/badges/64/413.png,5,4,0,1,8,4,4,12
5,15,Aston Villa,https://e0.365dm.com/football/badges/64/238.png,5,1,1,3,4,9,-5,4
5,16,Bournemouth,https://e0.365dm.com/football/badges/64/333.png,4,0,3,1,6,7,-1,3
5,4,Brentford,https://e0.365dm.com/football/badges/64/194.png,5,2,3,0,10,4,6,9
5,3,Brighton and Hove Albion,https://e0.365dm.com/football/badges/64/212.png,5,3,1,1,16,5,11,10
5,9,Chelsea,https://e0.365dm.com/football/badges/64/524.png,5,2,1,2,10,12,-2,7
5,18,Coventry City,https://e0.365dm.com/football/badges/64/571.png,5,1,0,4,1,10,-9,3
5,17,Crystal Palace,https://e0.365dm.com/football/badges/64/234.png,4,1,0,3,6,11,-5,3
5,5,Everton,https://e0.365dm.com/football/badges/64/229.png,5,2,3,0,6,3,3,9
5,20,Fulham,https://e0.365dm.com/football/badges/64/407.png,4,0,1,3,4,7,-3,1


In [0]:
display(spark.table(f"{catalog}.{db_name}.week_{week_value}_table").orderBy("position"))

week,position,team,logo,played,won,drawn,lost,goals_for,goals_against,goal_difference,points
5,1,Manchester City,https://e0.365dm.com/football/badges/64/345.png,4,4,0,0,8,2,6,12
5,2,Arsenal,https://e0.365dm.com/football/badges/64/413.png,5,4,0,1,8,4,4,12
5,3,Brighton and Hove Albion,https://e0.365dm.com/football/badges/64/212.png,5,3,1,1,16,5,11,10
5,4,Brentford,https://e0.365dm.com/football/badges/64/194.png,5,2,3,0,10,4,6,9
5,5,Everton,https://e0.365dm.com/football/badges/64/229.png,5,2,3,0,6,3,3,9
5,6,Leeds United,https://e0.365dm.com/football/badges/64/183.png,4,2,2,0,7,3,4,8
5,7,Hull City,https://e0.365dm.com/football/badges/64/253.png,5,2,2,1,6,4,2,8
5,8,Newcastle United,https://e0.365dm.com/football/badges/64/409.png,5,2,2,1,9,9,0,8
5,9,Chelsea,https://e0.365dm.com/football/badges/64/524.png,5,2,1,2,10,12,-2,7
5,10,Liverpool,https://e0.365dm.com/football/badges/64/155.png,4,1,3,0,6,4,2,6


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import from_csv, col
from pyspark.sql.window import Window

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.week_{week_value}_result (
    week int,
    match_date date,
    team_name string,
    opponent string,
    venue string,
    result_score string,
    result string,
    goal_scorers string,
    goal_times string,
    form_last_5 string
) USING DELTA
""")


DataFrame[]

In [0]:
result_clean = (spark.read.table(f"{catalog}.{db_name}.week_{week_value}_results_bz")
    .withColumn("week", F.col("week").cast("int"))
    .withColumn("match_date", F.to_date(F.trim(F.col("match_date")), "yyyy-MM-dd"))
    #.withColumn("match_date", F.date_format("match_date", "dd/MM/yyyy"))
    .select("week", "match_date", "team_name", "opponent", "venue", "result_score", "result", "goal_scorers", "goal_times",
            "form_last_5")
    .dropDuplicates(["team_name"])
)

display(result_clean)

week,match_date,team_name,opponent,venue,result_score,result,goal_scorers,goal_times,form_last_5
3,2026-09-06,Arsenal,Chelsea,H,2-1,W,Havertz 20'; Odegaard 50',20'; 50',WWW
3,2026-09-05,Aston Villa,Hull City,A,0-0,D,-,-,LLD
3,2026-09-05,Bournemouth,Newcastle United,A,2-2,D,Tavernier 9'; Thiaw (OG) 35',9'; 35',LDD
3,2026-09-05,Brentford,Sunderland,H,1-1,D,Janelt 57',57',WDD
3,2026-09-05,Brighton,Leeds United,H,1-1,D,Vuskovic 71',71',WLD
3,2026-09-06,Chelsea,Arsenal,A,1-2,L,Rogers 1',1',WWL
3,2026-09-05,Coventry City,Manchester City,A,0-1,L,-,-,LLL
3,2026-09-05,Crystal Palace,Fulham,A,3-2,W,Mitchell 35'; Mitchell 54'; Chilwell 77',35'; 54'; 77',LLW
3,2026-09-06,Everton,Manchester United,H,2-2,D,George 84'; Maitland-Niles 90',84'; 90',WDD
3,2026-09-05,Fulham,Crystal Palace,H,2-3,L,King 11'; Palacios 42',11'; 42',LLL


In [0]:

result_clean.createOrReplaceTempView("result_source")

spark.sql(f"""
MERGE INTO {catalog}.{db_name}.week_{week_value}_result r
USING result_source s
ON r.team_name = s.team_name
WHEN NOT MATCHED THEN INSERT *
""")

display(spark.table(f"{catalog}.{db_name}.week_{week_value}_result"))

week,match_date,team_name,opponent,venue,result_score,result,goal_scorers,goal_times,form_last_5
3,2026-09-06,Arsenal,Chelsea,H,2-1,W,Havertz 20'; Odegaard 50',20'; 50',WWW
3,2026-09-05,Aston Villa,Hull City,A,0-0,D,-,-,LLD
3,2026-09-05,Bournemouth,Newcastle United,A,2-2,D,Tavernier 9'; Thiaw (OG) 35',9'; 35',LDD
3,2026-09-05,Brentford,Sunderland,H,1-1,D,Janelt 57',57',WDD
3,2026-09-05,Brighton,Leeds United,H,1-1,D,Vuskovic 71',71',WLD
3,2026-09-06,Chelsea,Arsenal,A,1-2,L,Rogers 1',1',WWL
3,2026-09-05,Coventry City,Manchester City,A,0-1,L,-,-,LLL
3,2026-09-05,Crystal Palace,Fulham,A,3-2,W,Mitchell 35'; Mitchell 54'; Chilwell 77',35'; 54'; 77',LLW
3,2026-09-06,Everton,Manchester United,H,2-2,D,George 84'; Maitland-Niles 90',84'; 90',WDD
3,2026-09-05,Fulham,Crystal Palace,H,2-3,L,King 11'; Palacios 42',11'; 42',LLL


In [0]:
split = (
    spark.read.table(f"{catalog}.{db_name}.week_{week_value}_result")
    .withColumn("goals_for", F.split(F.col("result_score"), "-").getItem(0).cast("int"))
    .withColumn("goals_against", F.split(F.col("result_score"), "-").getItem(1).cast("int"))
    .select("week", "match_date", "team_name", "opponent", "venue", "result_score",
            "goals_for", "goals_against", "result", "goal_scorers", "goal_times",
            "form_last_5")
    .dropDuplicates(["team_name"])
)

display(split)

week,match_date,team_name,opponent,venue,result_score,goals_for,goals_against,result,goal_scorers,goal_times,form_last_5
2,2026-08-31,Arsenal,Aston Villa,A,1-0,1,0,W,Saka 59',59',WW
2,2026-08-31,Aston Villa,Arsenal,H,0-1,0,1,L,-,-,LL
2,2026-08-29,Bournemouth,Everton,H,1-1,1,1,D,Scott 41',41',LD
2,2026-08-30,Brentford,Leeds United,A,1-1,1,1,D,Schade 41',41',WD
2,2026-08-30,Brighton,Chelsea,A,3-4,3,4,L,Yalcouye 35'; Joao Pedro (OG) 63'; Gross (pen) 90',35'; 63'; 90',WL
2,2026-08-30,Chelsea,Brighton,H,4-3,4,3,W,Lavia 4'; Neto 14'; Joao Pedro 32'; Palmer 74',4'; 14'; 32'; 74',WW
2,2026-08-29,Coventry City,Hull City,H,0-1,0,1,L,-,-,LL
2,2026-08-28,Crystal Palace,Manchester City,H,1-4,1,4,L,Donnarumma (OG) 65',65',LL
2,2026-08-29,Everton,Bournemouth,A,1-1,1,1,D,Tarkowski 90',90',WD
2,2026-08-30,Fulham,Sunderland,A,0-1,0,1,L,-,-,LL
